# 08 — WHEN-OTHERWISE vs CASE-WHEN

**DataFrame API `when/otherwise`** ↔ **Spark SQL `CASE WHEN`** — deriving & overwriting columns, NULL gotchas, and multi-branch age categorization.

In [0]:
# Setup: SparkSession, employees DataFrame (NULLs + duplicate row), temp view for Spark SQL
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, expr, count
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("conditional-constructs").getOrCreate()

emp_data = [
    (1, "Manish", 26, 20000, "INDIA", "IT"),
    (2, "Rahul", None, 40000, "GERMANY", "ENGINEERING"),
    (3, "Pawan", 12, 60000, "INDIA", "SALES"),
    (4, "Roshini", 44, None, "UK", "ENGINEERING"),
    (5, "Raushan", 35, 70000, "INDIA", "SALES"),
    (6, None, 29, 200000, "UK", "IT"),
    (7, "Adam", 37, 65000, "US", "IT"),
    (8, "Chris", 16, 40000, "US", "SALES"),
    (None, None, None, None, None, None),
    (7, "Adam", 37, 65000, "US", "IT"),
]

schema = """
    emp_id INT,
    emp_name STRING,
    age INT,
    salary DOUBLE,
    country STRING,
    department STRING
"""
emp_df = spark.createDataFrame(data=emp_data, schema=schema)
emp_df.createOrReplaceTempView("employees_df")
emp_df.printSchema()
emp_df.show()



In [0]:
# Derive is_adult column — first matching branch wins, NULL age falls to otherwise('N/A')
(
    emp_df
    .withColumn("is_adult", F.when(col("age") < 18, "No")
                            .when(col("age") >= 18, "Yes")
                            .otherwise("N/A"))
).show()

In [0]:
# Spark SQL Equivalent — CASE WHEN for is_adult (NULL age falls to ELSE 'N/A')
spark.sql("""
    SELECT *,
        CASE 
            WHEN age < 18 THEN 'No'
            WHEN age >= 18 THEN 'Yes'
            ELSE 'N/A'
        END AS is_adult
    FROM employees_df       
""").show()

In [0]:
# expr() — write raw Spark SQL (CASE WHEN) inside the DataFrame API
# Middle ground between the DF cell above and the spark.sql() cell before it
(
    emp_df
    .withColumn("is_adult", expr("""
        CASE
            WHEN age < 18 THEN 'No'
            WHEN age >= 18 THEN 'Yes'
            ELSE 'N/A'
        END
    """))
).show()

In [0]:
# Overwrite the existing 'age' column — NULL-fill with 19, then derive is_adult

# Consider age = NULL as adult, with default value of 19 age
(
    emp_df
    .withColumn("age", F.when(col("age").isNull(), lit(19)).otherwise(col("age"))) # In this case if otherwise is missed all value will get set to NULL by default
    .withColumn("is_adult", F.when(col("age") < 18, "No").otherwise("Yes"))
).show()

In [0]:
# Simpler NULL-fill alternates — when(isNull...) is NOT the only way

# 1) F.coalesce(col, default) — exactly equivalent to when(col.isNull(), default).otherwise(col)
(
    emp_df
    .withColumn("age", F.coalesce(col("age"), lit(19)))
    .withColumn("is_adult", F.when(col("age") < 18, "No").otherwise("Yes"))
).show()

print("==============================================")

# 2) fillna — DataFrame-level NULL fill by column name
(
    emp_df
    .fillna({"age": 19})
    .withColumn("is_adult", F.when(col("age") < 18, "No").otherwise("Yes"))
).show()

In [0]:
# withColumns (plural) — derive MULTIPLE columns in one call (no chained withColumn)
# Also: a boolean column works directly as a when() condition — no need for '== True'
(
    emp_df
    .withColumn("high_earner", col("salary") > 50000)   # boolean column (NULL salary -> NULL flag)
    .withColumns({
        # boolean column used directly as the condition:
        "earner_flag": F.when(col("high_earner"), "High").otherwise("Normal"),
        "age_category": F.when((col("age") >= 0) & (col("age") < 18), "Minor")
                          .when((col("age") >= 18) & (col("age") < 60), "Adult")
                          .when(col("age") >= 60, "Senior Citizen")
                          .otherwise("Unknown"),
    })
).show()

In [0]:
# Variant 1 — CTE: NULL-fill age first, then derive is_adult
spark.sql("""
WITH cte AS (
    SELECT emp_id,
        CASE 
            WHEN age IS NULL THEN 19
            ELSE age
        END AS age,
        salary, 
        country,
        department
    FROM employees_df  
)

SELECT *, 
    CASE 
        WHEN age < 18 THEN "No"
        ELSE "Yes"
    END AS is_adult
FROM cte
""").show()

print("==============================================")

# Spark SQL Equivalent
spark.sql("""

    SELECT emp_id,
        CASE 
            WHEN age IS NULL THEN 19
            ELSE age
        END AS age,
        salary, 
        country,
        department,
        CASE
            WHEN age IS NULL OR age >= 18 THEN 'Yes'
            ELSE 'No'
        END AS is_adult
    FROM employees_df  

""").show()


In [0]:
# Categorize employees based on age, as follows
# Age 0-17  ==> Minor
# Age >=18  ==> Adult
# Age >=60  ==> Senior Citizen
# Otherwise ==> Invalid

(
    emp_df
    .withColumn(
        "age_category", 
            F.when( (col("age") >= 0) & (col("age") < 18), "Minor")
            .when((col("age") >= 18) & (col("age") < 60), "Adult")
            .when(col("age") >= 60, "Senior Citizen")
            .otherwise("Unknown")
    )
).show()

In [0]:
# Spark SQL Equivalent — CASE-WHEN for age_category (NULL ages fall to ELSE 'Unknown')
spark.sql("""
    SELECT *,
        CASE 
            WHEN age > 0 AND age < 18 THEN 'Minor'
            WHEN age >= 18 AND age < 60 THEN 'Adult'
            WHEN age >= 60 THEN 'Senior Citizen'
            ELSE 'Unknown'
        END AS age_category
    FROM employees_df
""").show()

In [0]:
# Simple CASE (SQL only) — match a VALUE directly instead of writing conditions
# NULL country never equals any WHEN value -> falls to ELSE 'Other'
spark.sql("""
    SELECT emp_name, country,
        CASE country
            WHEN 'INDIA' THEN 'APAC'
            WHEN 'US'    THEN 'NA'
            WHEN 'UK'    THEN 'EU'
            ELSE 'Other'
        END AS region
    FROM employees_df
""").show()

print("==============================================")

# CASE in ORDER BY — custom sort priority (INDIA first, then US, then the rest)
spark.sql("""
    SELECT emp_id, country
    FROM employees_df
    ORDER BY CASE country WHEN 'INDIA' THEN 0 WHEN 'US' THEN 1 ELSE 2 END, emp_id
""").show()

In [0]:
# Conditional aggregation — when() inside sum()/count() after a groupBy
# The classic reporting pattern: pivot-style metrics in one pass

# DataFrame API:
(
    emp_df
    .groupBy("department")
    .agg(
        F.sum(F.when(col("age") < 18, col("salary")).otherwise(lit(0))).alias("minor_salary"),
        F.sum(F.when(col("age") >= 18, col("salary")).otherwise(lit(0))).alias("adult_salary"),
        # count(when(...)) counts non-NULL results — non-matching rows yield NULL and are SKIPPED
        F.count(F.when(col("salary") > 50000, lit(1))).alias("high_earners"),
    )
).show()

print("==============================================")

# Spark SQL equivalent — SUM(CASE WHEN ...) as seen in real-world queries
spark.sql("""
    SELECT department,
        SUM(CASE WHEN age < 18 THEN salary ELSE 0 END) AS minor_salary,
        SUM(CASE WHEN age >= 18 THEN salary ELSE 0 END) AS adult_salary,
        COUNT(CASE WHEN salary > 50000 THEN 1 END)      AS high_earners
    FROM employees_df
    GROUP BY department
""").show()

In [0]:
# Edge cases — type coercion, age = 0, and NaN

# 1) Branch type coercion: mixing int and string branches -> ALL become STRING ('19', not 19)
(spark.createDataFrame([(1,)], "id INT")
    .select(F.when(col("id") == 1, lit(19)).otherwise(lit("adult")).alias("mixed_type"))
).show()

# 2) age = 0: Cell 7 uses >= 0 -> 'Minor', but Cell 8's SQL uses > 0 -> 'Unknown'
(spark.createDataFrame([(0,)], "age INT")
    .select(
        F.when((col("age") >= 0) & (col("age") < 18), "Minor").otherwise("Unknown").alias("df_api_ge_0"),
        expr("CASE WHEN age > 0 AND age < 18 THEN 'Minor' ELSE 'Unknown' END").alias("sql_gt_0"),
    )
).show()

# 3) NaN is NOT like NULL in comparisons — Spark treats NaN as GREATER than every number,
#    so NaN >= 18 is TRUE and NaN rows silently match your '>=' branches.
#    isNull() does NOT catch NaN — use isnan(). (Note the NaN row got 'Yes' below!)
nan_df = spark.createDataFrame([(float("nan"),), (28.0,), (None,)], "age FLOAT")
(
    nan_df
    .withColumn("is_adult", F.when(col("age") >= 18, "Yes").otherwise("No"))
    .withColumn("null_check", F.when(col("age").isNull(), "NULL").otherwise("not NULL"))
    .withColumn("is_nan", F.isnan(col("age")))
).show()

# Quick Revision Notes (Handwritten-Style) — WHEN-OTHERWISE & CASE-WHEN

## 1 — Syntax map

* DF API: `F.when(cond, val).when(cond, val).otherwise(val)`
* SQL: `CASE WHEN c THEN v WHEN c THEN v ELSE v END`
* Both evaluate **top-down, first match wins** — remaining branches skipped

## 2 — NULL is the biggest gotcha ⚠️

* NULL in any comparison (`age < 18`, `age >= 18`) → result is NULL, NOT true
* So NULL age does NOT match any `when` → lands in `.otherwise()` / `ELSE`
* Cell 3: NULL age → `N/A` (caught by otherwise), not "No" or "Yes"
* 💡 Handle NULLs explicitly, BEFORE range checks:
  `F.when(col("age").isNull(), default).otherwise(col("age"))`

## 3 — Missing `.otherwise()` → unmatched rows become NULL

* Same as SQL `CASE` with no `ELSE` → NULL
* Cell 5 comment: drop the `.otherwise(col("age"))` → ALL ages become NULL

## 4 — Overwrite an existing column

* `withColumn("age", ...)` with the SAME name replaces it in place
* Null-fill pattern: `F.when(col("age").isNull(), lit(19)).otherwise(col("age"))`
* Two `withColumn` calls in a chain → 2nd sees the NEW age (19 → "Yes")

## 5 — Multi-branch categorization (Cell 7 / 8)

* Age 0–17 → Minor, 18–59 → Adult, ≥60 → Senior Citizen, else Unknown
* Use exclusive ranges: `(age >= 0) & (age < 18)`
* ⚠️ Cell 8 SQL uses `age > 0` (not `>= 0`) → age = 0 gives 'Unknown' there

## 6 — Alternate usages (new cells)

* `expr("CASE WHEN ... END")` — raw SQL inside DF API
* `F.coalesce(col, lit(19))` == `when(col.isNull(), lit(19)).otherwise(col)`
* `df.fillna({"age": 19})` — DataFrame-level NULL fill
* `withColumns({...})` — several columns in ONE call
* Boolean column as condition: `when(col("flag"), v)` — no `== True` needed
* Simple CASE (SQL): `CASE country WHEN 'INDIA' THEN 'APAC' ...` — matches VALUES
* CASE in ORDER BY — custom sort priority (`CASE ... THEN 0/1/2 END`)
* Conditional aggregation: `F.sum(F.when(cond, salary).otherwise(lit(0)))` = SQL `SUM(CASE WHEN ...)`
* `F.count(F.when(cond, lit(1)))` — non-matching rows yield NULL and are NOT counted

## 7 — Edge cases (new cells)

* Branch type coercion: mixing `19` and `"adult"` → ALL branches become STRING
* age = 0: `>= 0` → Minor, `> 0` → Unknown (Cell 7 vs Cell 8 mismatch)
* NaN is NOT caught by `isNull()` — and unlike NULL, Spark compares NaN as GREATER than every number, so NaN rows silently match `>=` branches (see the NaN demo: `NaN >= 18` → TRUE → 'Yes')

## DF API vs SQL

| DataFrame API | Spark SQL |
| --- | --- |
| `F.when(c, v)` | `WHEN c THEN v` |
| chained `.when(...)` | more `WHEN` lines |
| `.otherwise(v)` | `ELSE v` |
| no `.otherwise()` → NULL | no `ELSE` → NULL |
| `&` / `\|` joins (wrap in `()`) | `AND` / `OR` |
| `col("x").isNull()` | `x IS NULL` |
| `col("x").isNotNull()` | `x IS NOT NULL` |

## Quick Recall Box ⭐

* WHEN = when, OTHERWISE = ELSE — same engine underneath
* First matching branch wins, rest skipped
* NULL never matches a comparison → handle it first or it slips to otherwise/ELSE
* No otherwise/ELSE → unmatched = NULL (dangerous default!)
* `&` needs parentheses: `(c1) & (c2)`, not `c1 & c2`
* `withColumn("same_name", ...)` overwrites; later links see updated value
* `coalesce()`/`fillna()` beat `when(isNull())` for plain NULL-fills
* `sum(when(...))` = conditional aggregation; `count(when(...))` counts matches only
* `isnan()` catches NaN, `isNull()` catches NULL — different functions, both needed; NaN silently matches `>=` branches
* All branch values coerce to ONE type — mixed int/string → all STRING